# Khóa prompt: bộ prompt tách hướng `ban5-en-tach-p3` trên Qwen3 (NLP_N12)

Notebook chạy trên **Kaggle, GPU T4 x2**. Nó gán cả hai bộ gold (480 câu) bằng một mô hình Qwen3 với prompt định dùng để gán toàn bộ dataset, rồi chấm. Kết quả dùng để chọn mô hình và quyết định khóa prompt hay sửa tiếp.
- **Mô hình:** chọn bằng biến `MO_HINH` ở ô 4. Mặc định là `"14b"`.
  - `"14b"`: `Qwen/Qwen3-14B-AWQ`, mô hình đã chọn: với p2 nó hơn 30B-A3B ở 3/4 hướng, cả hai vòng gold.
  - `"30b-a3b"`: `Qwen/Qwen3-30B-A3B-GPTQ-Int4`, đã chạy với p2 ngày 27/11. Chạy được trên 2 × T4, mỗi câu 1,38 giây.
  - Notebook không tự chuyển sang mô hình khác khi lỗi. Tên mô hình có trong tên file kết quả.
  - Tắt chế độ suy nghĩ, nhiệt độ 0.
- **Prompt:** `scripts/llm/prompt_ban5_en_tach/` bản `ban5-en-tach-p3`, mỗi câu gọi 4 lần (mỗi hướng một lần). p3 chỉ sửa toxicity:
  - mô hình chép nguyên văn từ xúc phạm hoặc từ tục vào trường `bang_chung`;
  - script kiểm tra bằng chứng: attack mà `bang_chung` không có trong câu, hoặc profane mà câu không có từ tục nào trong danh sách `chung.TU_TUC`, thì hỏi lại một lần; vẫn không có thì hạ nhãn (ghi ở cột `da_sua`);
  - thêm ví dụ tự đặt về từ không phải lời xúc phạm.
- **Gold:** vòng 2 là tập chỉnh prompt; vòng 1 là tập kiểm tra, chưa dùng để sửa prompt lần nào. Script chấm báo κ riêng từng vòng, và κ khi bỏ các câu gold trùng ví dụ trích trong prompt (20 câu vòng 2, 5 câu vòng 1).

**Tiêu chí khóa prompt** (xem ở ô 7):
1. Trên gold vòng 2, sau khi bỏ câu trùng: toxicity, topic ≥ 0,67; stance, emotion_1 ≥ 0,5.
2. Gold vòng 1 không thấp hơn vòng 2 quá khoảng 0,1 ở hướng nào. Thấp hơn nhiều nghĩa là prompt đang khớp riêng các câu vòng 2.
3. Không câu nào có lỗi, tốc độ đủ để gán 10.000 câu trong một phiên Kaggle (dưới khoảng 9 giờ).

Đạt cả ba thì khóa mô hình và prompt để gán toàn bộ. Chưa đạt thì chỉ sửa prompt theo sheet `cau_lech` (sheet này chỉ có câu vòng 2), không xem câu lệch của vòng 1.

**Trước khi chạy**
1. **Phải tải gói mới**: p3 đổi cả prompt lẫn code (`chung.py`, `gan_nhan_llm.py`). Trên máy chạy `python scripts/llm/dong_goi_kaggle.py`, rồi Kaggle › dataset cũ › **New Version**, tải `data/llm/goi_kaggle/goi_kaggle.zip` lên, để **Private**. Ô 2 kiểm tra đúng bản p3.
2. Import notebook này. Settings: **GPU T4 x2**, **Internet On**, Add Input là dataset trên.
3. Bấm **Save Version › Save & Run All (Commit)** để chạy nền; đóng trình duyệt vẫn chạy và file Output được giữ lại. Ước tính với 14B:
   - cài vLLM khoảng 5 phút;
   - tải mô hình (khoảng 10 GB) và khởi động khoảng 5–10 phút;
   - gán 480 câu × 4 lần gọi khoảng 15–20 phút.

   Đây là ước tính; thời gian thật in ở ô 7.
4. Tải `ket_qua_khoa_prompt_<mô hình>.zip` từ tab Output của phiên bản đã chạy, giải nén vào **thư mục gốc repo** (zip đã chứa đường dẫn `data/llm/...`).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L
import torch
N_GPU = torch.cuda.device_count()
print("Số GPU:", N_GPU)
assert N_GPU >= 2, "Cần 2 GPU: Settings > Accelerator > GPU T4 x2 (Qwen3-30B-A3B 4-bit không vừa một T4)"

## 2. Lấy gói dữ liệu và code
Tìm `goi_kaggle.zip` (hoặc bản Kaggle đã giải nén) trong `/kaggle/input`, chép vào `/kaggle/working/nlp`, kiểm tra đúng prompt p3.

In [ ]:
import glob, os, re, shutil, zipfile
WORK = "/kaggle/working/nlp"
zips = glob.glob("/kaggle/input/**/goi_kaggle.zip", recursive=True)
if zips:
    shutil.rmtree(WORK, ignore_errors=True)
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    found = glob.glob("/kaggle/input/**/scripts/llm/gan_nhan_llm.py", recursive=True)
    assert found, "Không thấy goi_kaggle.zip hay thư mục đã giải nén trong /kaggle/input: kiểm tra Add Input"
    src = found[0].rsplit("/scripts/llm/", 1)[0]
    shutil.rmtree(WORK, ignore_errors=True)
    shutil.copytree(src, WORK)
os.chdir(WORK)
PROMPT = "scripts/llm/prompt_ban5_en_tach"
ver = re.search(r"PROMPT_VERSION:\s*(\S+)", open(f"{PROMPT}/chung.md", encoding="utf-8").read()).group(1)
print("Prompt:", ver)
assert ver == "ban5-en-tach-p3", "Gói cũ: chạy lại dong_goi_kaggle.py trên máy và tải bản mới lên (New Version)"
assert "kiem_tra_bang_chung" in open("scripts/llm/chung.py", encoding="utf-8").read(), "Gói cũ: chung.py chưa có kiểm tra bằng chứng"
!find . -type f | sort

## 3. Cài vLLM
Nếu báo xung đột phiên bản torch hay CUDA: Run › Restart & clear cell outputs rồi chạy lại từ đầu. Lần trước chạy được với vLLM 0.34.0; nếu bản mới nhất lỗi thì đổi thành `vllm==0.34.0`.

In [ ]:
!pip install -q -U vllm openai 2>&1 | tail -3
!python -c "import vllm, openai; print('vllm', vllm.__version__, '| openai', openai.__version__)"

## 4. Khởi động máy chủ vLLM

Đổi `MO_HINH` để chọn mô hình. Ô thử lần lượt các cấu hình của mô hình đó cho đến khi máy chủ trả lời, rồi gửi một câu hỏi thử.
- **`--quantization awq`:** dùng cho Qwen3-14B-AWQ, như lần chạy trước. Với 30B-A3B, vLLM tự đọc kiểu GPTQ từ checkpoint.
- **`--dtype half`:** T4 không có bfloat16.
- **`--enable-expert-parallel`** (chỉ cấu hình 2 của 30B-A3B): chia nguyên các expert cho hai GPU. Lần trước không cần, vì cấu hình 1 đã chạy được.
- **Log:** mỗi lần thử ghi vào `/kaggle/working/vllm_<số>.log`. Lỗi thì ô in 3.000 ký tự cuối của log.

In [ ]:
import json, signal, subprocess, time, urllib.request

MO_HINH = "14b"   # "14b" hoặc "30b-a3b"
CAU_HINH = {
    # (tên Hugging Face, tên phục vụ (vào tên file kết quả), tham số thêm)
    "14b": [("Qwen/Qwen3-14B-AWQ", "qwen3-14b", ["--quantization", "awq"])],
    "30b-a3b": [("Qwen/Qwen3-30B-A3B-GPTQ-Int4", "qwen3-30b-a3b", []),
                ("Qwen/Qwen3-30B-A3B-GPTQ-Int4", "qwen3-30b-a3b", ["--enable-expert-parallel"])],
}
UNG_VIEN = CAU_HINH[MO_HINH]


def hoi_thu(served):
    body = {"model": served, "temperature": 0, "max_tokens": 20,
            "messages": [{"role": "user", "content": "Trả lời đúng một từ: OK"}],
            "chat_template_kwargs": {"enable_thinking": False}}
    req = urllib.request.Request("http://localhost:8000/v1/chat/completions", json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=120))["choices"][0]["message"]["content"]


def dung(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(60)
    except Exception:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except Exception:
            pass
    time.sleep(10)   # chờ giải phóng bộ nhớ GPU


def khoi_dong(i, model, served, extra, gioi_han=2400):
    cmd = ["vllm", "serve", model, "--dtype", "half", "--tensor-parallel-size", "2", "--disable-custom-all-reduce",
           "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--enable-prefix-caching",
           "--served-model-name", served, "--port", "8000", "--seed", "0"] + extra
    print(f"[{i}] " + " ".join(cmd))
    path = f"/kaggle/working/vllm_{i}.log"
    proc = subprocess.Popen(cmd, stdout=open(path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < gioi_han:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5).read()
        except Exception:
            time.sleep(15)
            continue
        try:
            tra_loi = hoi_thu(served)
        except Exception as e:
            tra_loi = f"lỗi: {e}"
        print(f"[{i}] sẵn sàng sau {time.time() - t0:.0f} giây; hỏi thử: {tra_loi!r}")
        if "ok" in tra_loi.lower():
            return proc, path
        print(f"[{i}] câu trả lời thử không đúng, bỏ cấu hình này")
        break
    print(f"[{i}] không khởi động được. Cuối log:")
    print(open(path, encoding="utf-8", errors="replace").read()[-3000:])
    dung(proc)
    return None, path


server = None
for i, (MODEL, SERVED, EXTRA) in enumerate(UNG_VIEN, 1):
    server, VLLM_LOG = khoi_dong(i, MODEL, SERVED, EXTRA)
    if server:
        break
assert server, "Không cấu hình nào khởi động được: xem các log /kaggle/working/vllm_*.log"
print(f"\n==> ĐANG CHẠY: {MODEL} {' '.join(EXTRA)} (tên phục vụ {SERVED})")

## 5. Chạy thử 5 câu
Xem 5 dòng: nhãn hợp lệ, cột `loi` trống, `ly_do` là tiếng Anh và có đủ 4 phần `toxicity: ... | topic: ... | stance: ... | emotion: ...`, không có đoạn `<think>`. Nếu `loi` báo `JSON hỏng` hay `max_tokens`, dừng lại xem log.

In [ ]:
GOLD = "1 2"
GAN = (f"python scripts/llm/gan_nhan_llm.py --gold {GOLD} --nha-cung-cap vllm --model {SERVED} --nhiet-do 0 "
       f"--luong 16 --khong-suy-nghi --prompt {PROMPT}")
!{GAN} --thu | head -3
!{GAN} --gioi-han 5 --ra /kaggle/working/thu_5_cau.jsonl > /dev/null
import pandas as pd
pd.set_option("display.max_colwidth", 200)
thu = pd.read_json("/kaggle/working/thu_5_cau.jsonl", lines=True)
display(thu[["toxicity", "topic", "stance", "emotion_1", "emotion_2", "hard", "ly_do", "vi_pham", "da_sua", "loi"]])
assert (thu.loi == "").all(), "Có câu lỗi khi chạy thử: xem cột loi và log vLLM"

## 6. Gán 480 câu gold
Chạy lại ô này sau khi bị ngắt thì script chỉ gán các câu còn thiếu hoặc lỗi.

In [ ]:
!{GAN}

## 7. Chấm so với gold, đo tốc độ

Mốc so sánh là lần chạy 14B với p2. p3 chỉ sửa toxicity, nên topic, stance, emotion chỉ nên dao động khoảng ±0,02.

| κ (bỏ câu trùng prompt) | toxicity | topic | stance | emotion_1 |
|---|---|---|---|---|
| Qwen3-14B, p2, vòng 2 | 0,480 | 0,714 | 0,488 | 0,574 |
| Qwen3-14B, p2, vòng 1 | 0,646 | 0,725 | 0,513 | 0,527 |
| Mục tiêu | 0,67 | 0,67 | 0,5 | 0,5 |

Ô in thêm số câu bị hỏi lại toxicity và số câu bị hạ nhãn.

Tốc độ lần trước: 14B p2 2,07 giây mỗi câu. p3 hỏi lại thêm một lần với các câu thiếu bằng chứng, nên có thể chậm hơn một chút.

Báo cáo Excel: `data/llm/danh_gia_khoa_prompt_<mô hình>_<phiên bản prompt>.xlsx`.

In [ ]:
KQ = f"data/llm/ket_qua/vllm_{SERVED}_mac-dinh_{ver}_gold12"
!python scripts/llm/danh_gia_llm.py --gold {GOLD} --kq {KQ}.jsonl --bo-trung-prompt {PROMPT} --ra data/llm/danh_gia_khoa_prompt_{SERVED}_{ver}.xlsx 2>/dev/null
s = json.load(open(f"{KQ}.tong_ket.json", encoding="utf-8"))
n = max(s["so_cau"], 1)
giay_cau = s["giay"] / n
print(f"\n{SERVED} · {s['prompt_version']}: {s['so_cau']} câu, lỗi {s['loi']}, vi phạm {s['vi_pham']}, hard {s['hard']}, {s['giay']} giây")
print(f"Tốc độ: {giay_cau:.2f} giây mỗi câu (4 lần gọi, 16 luồng) → 10.000 câu ≈ {giay_cau * 10000 / 3600:.1f} giờ; "
      f"toàn bộ 136.609 câu ≈ {giay_cau * 136609 / 3600:.0f} giờ GPU (Kaggle cho 30 giờ mỗi tuần)")
print("Lần trước: Qwen3-14B p2 2,07 giây mỗi câu; Qwen3-30B-A3B p2 1,38 giây mỗi câu.")
kq = pd.read_json(f"{KQ}.jsonl", lines=True).drop_duplicates("comment_uid", keep="last")
hoi_lai = kq.da_sua.str.contains("hỏi lại toxicity", na=False)
ha_nhan = kq.da_sua.str.contains("→", na=False)
print(f"Kiểm tra bằng chứng: hỏi lại toxicity {int(hoi_lai.sum())} câu, hạ nhãn {int(ha_nhan.sum())} câu")
print(kq[ha_nhan].da_sua.str.extract(r"toxicity (\w+ → \w+)")[0].value_counts().to_string())

## 8. Đóng gói kết quả để tải về, dọn dữ liệu
Tải `ket_qua_khoa_prompt_<mô hình>.zip` từ tab Output rồi giải nén vào thư mục gốc repo. Ô cuối dừng máy chủ và xóa bản dữ liệu đã giải nén.

In [ ]:
shutil.copy(VLLM_LOG, f"data/llm/vllm_khoa_prompt_{SERVED}_{ver}.log")
shutil.make_archive(f"/kaggle/working/ket_qua_khoa_prompt_{SERVED}", "zip", WORK, "data/llm")
print(zipfile.ZipFile(f"/kaggle/working/ket_qua_khoa_prompt_{SERVED}.zip").namelist())

In [ ]:
dung(server)
os.chdir("/kaggle/working")
shutil.rmtree(WORK, ignore_errors=True)
for f in glob.glob("/kaggle/working/thu_5_cau.jsonl") + glob.glob("/kaggle/working/vllm_*.log"):
    os.remove(f)
print("Đã xóa dữ liệu trong /kaggle/working/nlp; còn lại:", os.listdir("/kaggle/working"))